# Project 4 — Word Embeddings & the Embedding Layer

## What is a word embedding?
A **word embedding** is a *dense* numeric vector (typically 50–300 numbers) that represents the **meaning** of a word.
Words with similar meaning end up *close* in vector space.

Famous example:
$$\vec{king} - \vec{man} + \vec{woman} \approx \vec{queen}$$

## Why are embeddings better than TF-IDF?

| | TF-IDF | Embeddings |
|---|---|---|
| Captures meaning? | ❌ no | ✅ yes |
| Vector size | 10 000 – 100 000 | 50 – 300 |
| Sparse / dense | sparse | dense |
| Trainable? | no | yes (learned by the network) |

## What is an Embedding **Layer** in Keras?
It's a lookup table: **input** = integer ID of a word, **output** = its dense vector.
These vectors are *learned* during training along with the rest of the model.

```
word_id = 7  →  Embedding Layer  →  [0.21, -0.07, 0.93, …]   (shape: embed_dim)
```

## Part A — Pre-trained embeddings via spaCy

*(For real word vectors install a larger spaCy model: `python -m spacy download en_core_web_md`)*

In [6]:
import numpy as np, spacy
nlp = spacy.load('en_core_web_sm')

words = ['king', 'queen', 'man', 'woman', 'apple', 'banana', 'computer']
for w in words:
    token = nlp(w)[0]
    print(f'{w:<10} vector shape = {token.vector.shape}')

king       vector shape = (96,)
queen      vector shape = (96,)
man        vector shape = (96,)
woman      vector shape = (96,)
apple      vector shape = (96,)
banana     vector shape = (96,)
computer   vector shape = (96,)


### Word similarity
spaCy computes cosine similarity between two vectors. (Use `en_core_web_md` for meaningful values.)

In [7]:
for w1, w2 in [('king', 'queen'), ('king', 'apple'),
                ('man', 'woman'), ('dog', 'cat'), ('computer', 'banana')]:
    s = nlp(w1)[0].similarity(nlp(w2)[0])
    print(f'  {w1:<8} ↔ {w2:<8}  similarity = {s:.4f}')

  king     ↔ queen     similarity = 0.4220
  king     ↔ apple     similarity = 0.6904
  man      ↔ woman     similarity = 0.8134
  dog      ↔ cat       similarity = 0.7423
  computer ↔ banana    similarity = 0.7288


C:\Users\Vineet Tiwari\AppData\Local\Temp\ipykernel_16304\3151031878.py:3: UserWarning: [W007] The model you're using has no word vectors loaded, so the result of the Token.similarity method will be based on the tagger, parser and NER, which may not give useful similarity judgements. This may happen if you're using one of the small models, e.g. `en_core_web_sm`, which don't ship with word vectors and only use context-sensitive tensors. You can always add your own word vectors, or use one of the larger models instead if available.
  s = nlp(w1)[0].similarity(nlp(w2)[0])


## Part B — Learning embeddings inside a Keras model

We'll build a tiny sentiment classifier where the network *learns* an embedding for every word.

In [8]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GlobalAveragePooling1D, Dense

sentences = [
    'I love this movie', 'This film was great', 'Best movie ever',
    'Loved every second', 'What a fantastic story',
    'I hated this film', 'Worst movie ever', 'Boring and dull',
    'Terrible acting and plot', 'I do not like this movie',
]
labels = np.array([1]*5 + [0]*5)
labels

array([1, 1, 1, 1, 1, 0, 0, 0, 0, 0])

### Tokenize and pad
Each unique word is assigned an integer ID; sequences are padded to the same length.

In [10]:
VOCAB_SIZE, MAX_LEN, EMBED_DIM = 50, 6, 8

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')
tokenizer.fit_on_texts(sentences)

sequences = tokenizer.texts_to_sequences(sentences)
X = pad_sequences(sequences, maxlen=MAX_LEN, padding='post')

print('Word index (first 10):')
for w, i in list(tokenizer.word_index.items())[:10]:
    print(f'  {w:<12} → {i}')
print('\nPadded matrix:\n', X)

Word index (first 10):
  <OOV>        → 1
  this         → 2
  movie        → 3
  i            → 4
  film         → 5
  ever         → 6
  and          → 7
  love         → 8
  was          → 9
  great        → 10

Padded matrix:
 [[ 4  8  2  3  0  0]
 [ 2  5  9 10  0  0]
 [11  3  6  0  0  0]
 [12 13 14  0  0  0]
 [15 16 17 18  0  0]
 [ 4 19  2  5  0  0]
 [20  3  6  0  0  0]
 [21  7 22  0  0  0]
 [23 24  7 25  0  0]
 [ 4 26 27 28  2  3]]


### Build the model

**Layer cheat-sheet:**
- `Embedding(input_dim, output_dim, input_length)` — vocab → vectors
- `GlobalAveragePooling1D()` — average all word vectors → one sentence vector
- `Dense(1, sigmoid)` — output 0/1 probability

In [ ]:
model = Sequential([
    Embedding(VOCAB_SIZE, EMBED_DIM, input_length=MAX_LEN, name='embedding_layer'),
    GlobalAveragePooling1D(),
    Dense(8, activation='relu'),
    Dense(1, activation='sigmoid'),
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

d:\Deep_Learning\NLP-Project\venv\Lib\site-packages\keras\src\layers\core\embedding.py:103: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_layer (Embedding)     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ ?                      │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

### Train

In [12]:
history = model.fit(X, labels, epochs=50, verbose=0)
print(f'Final training accuracy: {history.history["accuracy"][-1]:.2f}')

Final training accuracy: 0.90


### Inspect the learned embeddings
After training, every word in the vocabulary has its own learned vector.

In [13]:
embedding_matrix = model.get_layer('embedding_layer').get_weights()[0]
print('Embedding matrix shape:', embedding_matrix.shape)

movie_idx = tokenizer.word_index['movie']
print(f"Vector for 'movie' (index {movie_idx}):")
print(embedding_matrix[movie_idx])

Embedding matrix shape: (50, 8)
Vector for 'movie' (index 3):
[-0.02476292  0.05420351 -0.00775182 -0.03817471  0.04099883  0.02414984
 -0.01981453 -0.02209395]


### Predict on new reviews

In [14]:
new_reviews = ['I love this story', 'This was terrible and boring']
new_X = pad_sequences(tokenizer.texts_to_sequences(new_reviews), maxlen=MAX_LEN)
for r, p in zip(new_reviews, model.predict(new_X, verbose=0)):
    s = 'POSITIVE' if p[0] > 0.5 else 'NEGATIVE'
    print(f'  "{r}" → {s} (score={p[0]:.2f})')

  "I love this story" → POSITIVE (score=0.51)
  "This was terrible and boring" → NEGATIVE (score=0.49)


## Summary

- **Pre-trained embeddings** (Word2Vec, GloVe, FastText, spaCy) — quick to use, generic meaning.
- **Learned embeddings** (Keras `Embedding` layer) — tailored to your task, need more data.
- The Embedding layer is the foundation of *every* modern NLP model (LSTM, GRU, Transformer, BERT, GPT).